# Assignment 08: Grouped Summaries of an Imaging Network

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import pyarrow

print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

MISSING_PATH = OUTPUT_DIR / "missing_by_modality.csv"
SITE_SUMMARY_PATH = OUTPUT_DIR / "site_summary.csv"
SITE_MODALITY_PATH = OUTPUT_DIR / "site_modality_summary.csv"
DELAYED_STAT_PATH = OUTPUT_DIR / "delayed_stat_exams.csv"
CT_DOSE_PATH = OUTPUT_DIR / "ct_dose_review.csv"
SLOWEST_PATH = OUTPUT_DIR / "slowest_stat_reports.csv"
PIVOT_PATH = OUTPUT_DIR / "median_report_pivot.csv"
PAIRPLOT_PNG = OUTPUT_DIR / "pairplot.png"  # seaborn route
PAIRPLOT_HTML = OUTPUT_DIR / "pairplot.html"  # Altair route: one of these two
PAIRPLOT_JSON = OUTPUT_DIR / "pairplot.json"

## Load the data

This cell downloads the exam log from the course repository when `data/` lacks it (`urlretrieve(url, path)` saves the file at a web address to `path`), then reads it: one row per exam. Run it as it is.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/christopherseaman/datasci_217/main/08/assignment/data"
DATA_FILES = ["imaging_exams.parquet"]

DATA_DIR.mkdir(exist_ok=True)
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        print("downloading", name)
        urlretrieve(f"{DATA_URL}/{name}", DATA_DIR / name)

exams = pd.read_parquet(DATA_DIR / "imaging_exams.parquet")
print("exams:", exams.shape)
exams.head()

## Task 1: Explore the log and summarize each site

### 1.1 Explore the exam log

> **Checkpoint: `output/pairplot.png`, or `output/pairplot.html` or `.json`**
> One pairplot of all five numeric columns, from seaborn or Altair.

> **Checkpoint: `output/missing_by_modality.csv`**
> Four rows, `CT` to `X-ray`, under the header `modality,report_hr,dose_msv`.

In [ ]:
distinct_exams = exams["exam_id"].nunique()  # TODO (README 1.1 step 1): distinct exam_id values
patients = exams["patient_id"].nunique()  # TODO (README 1.1 step 1): distinct patient_id values
print("rows:", len(exams))
print("distinct exams:", distinct_exams)
print("patients:", patients)

# TODO (README 1.1 step 2): info() and describe()
info = exams.info()
describe = exams.describe()
display(info)
display(describe)

# Numeric columns: "age", "scan_min", "dose_msv", "wait_hr", "report_hr"


In [ ]:
pairplot_columns = ["age", "scan_min", "dose_msv", "wait_hr", "report_hr"] # TODO (README 1.1 step 3): the five numeric columns
pairplot_sample = exams[pairplot_columns].sample(1000, random_state=1000)  # TODO (README 1.1 step 3): those columns for a random sample of 1,000 exams

display(pairplot_columns)
display(pairplot_sample)

# TODO (README 1.1 step 3): pick one route and delete the other
# seaborn: import seaborn as sns, draw the pairplot, and save it to PAIRPLOT_PNG
import seaborn as sns
grid = sns.pairplot(pairplot_sample, corner = True)
grid.savefig(PAIRPLOT_PNG)


In [ ]:
missing_by_modality = exams[["report_hr","dose_msv"]].isna().groupby(exams["modality"]).mean()  # TODO (README 1.1 step 4): share of exams with no report_hr and no dose_msv, by modality
display(missing_by_modality)

# TODO (README 1.1 step 5): save missing_by_modality to MISSING_PATH, keeping its modality index
missing_by_modality.to_csv(MISSING_PATH)

q1, q3 = exams["wait_hr"].quantile([0.25,0.75]) # TODO (README 1.1 step 6): quartiles of wait_hr
upper = q3 + 1.5*(q3-q1)  # TODO (README 1.1 step 6): the IQR upper fence
above_fence = sum(exams["wait_hr"] > upper)  # TODO (README 1.1 step 6): how many waits are above it
print("quartiles:", q1, q3)
print("upper fence:", upper)
print("waits above the fence:", above_fence)

### 1.2 Put the sites in reporting order

This step saves nothing.

In [ ]:
SITE_ORDER = ["Downtown", "Harbor", "Northgate", "Lakeside", "Hillcrest", "Riverside", "Westfield", "Valley", "Eastport"]  # Eastport opens in 2026

# TODO (README 1.2 step 1): overwrite exams["site"] with an ordered categorical in SITE_ORDER
exams["site"] = pd.Categorical(exams["site"], categories = SITE_ORDER)
print("site dtype:", exams["site"].dtype)

exams_per_site = exams.groupby("site", observed=False, dropna=False)["exam_id"].count()  # TODO (README 1.2 step 2): exams per site, keeping Eastport
exams_per_site

### 1.3 Count exams, reports, and patients, and total the scanner time

> **Checkpoint: `output/site_summary.csv`**
> Eight rows under the header `site,exam_count,report_count,patient_count,total_scan_min,mean_scan_min`.

In [ ]:
# TODO (README 1.3 step 1): one groupby on site with five named aggregations
site_summary = exams.groupby("site", as_index=False).agg(
    exam_count = ("exam_id", "count"),
    report_count = ("report_hr", "count"),
    patient_count = ("patient_id", "nunique"),
    total_scan_min = ("scan_min", "sum"),
    mean_scan_min = ("scan_min", "mean")
)  
print("rows:", len(site_summary))

# TODO (README 1.3 step 2): save site_summary to SITE_SUMMARY_PATH
site_summary.to_csv(SITE_SUMMARY_PATH, index=False)
site_summary

## Task 2: Compare sites, modalities, and single exams

### 2.1 Summarize each site and modality

> **Checkpoint: `output/site_modality_summary.csv`**
> 31 rows under the header `site,modality,exam_count,median_wait_hr,median_report_hr`.

In [ ]:
# TODO (README 2.1 step 1): group by site and modality with three named aggregations
site_modality = exams.groupby(["site", "modality"], as_index=False).agg(
    exam_count = ("exam_id", "count"),
    median_wait_hr = ("wait_hr", "median"),
    median_report_hr = ("report_hr", "median")
)    
print("rows:", len(site_modality))

# TODO (README 2.1 step 2): save site_modality to SITE_MODALITY_PATH
site_modality.to_csv(SITE_MODALITY_PATH,index=False)
site_modality

### 2.2 Flag STAT exams that waited far longer than their peers

> **Checkpoint: `output/delayed_stat_exams.csv`**
> 190 rows under the header `exam_id,site,modality,wait_hr,peer_median_wait_hr,wait_vs_peer_hr`.

In [ ]:
stat = exams[exams["priority"] == "STAT"].copy()  # TODO (README 2.2 step 1): the STAT exams, as a copy

# TODO (README 2.2 step 2): add peer_median_wait_hr with transform
stat["peer_median_wait_hr"] = stat.groupby(["site", "modality"])["wait_hr"].transform("median")
stat_delayed = stat[stat["wait_hr"] > stat["peer_median_wait_hr"]]
display(stat_delayed)

# TODO (README 2.2 step 3): add wait_vs_peer_hr
stat["wait_vs_peer_hr"] = stat["wait_hr"] - stat["peer_median_wait_hr"]
display(stat["wait_vs_peer_hr"])
display(stat)
print("STAT exams:", len(stat))

delayed_stat = stat[stat["wait_hr"] > 8]  # TODO (README 2.2 step 4): STAT exams that waited over 8 hours, six columns
delayed_stat = delayed_stat[["exam_id", "site", "modality", "wait_hr", "peer_median_wait_hr", "wait_vs_peer_hr"]]
print("delayed:", len(delayed_stat))

# TODO (README 2.2 step 5): save delayed_stat to DELAYED_STAT_PATH
delayed_stat.to_csv(DELAYED_STAT_PATH, index=False)
delayed_stat.sort_values("wait_hr", ascending=False).head(10)

### 2.3 Build a CT radiation-dose review list

> **Checkpoint: `output/ct_dose_review.csv`**
> 1,253 rows under the header `exam_id,patient_id,dose_msv`.

In [ ]:
ct = exams[exams["modality"] == "CT"].copy() # TODO (README 2.3 step 1): the CT exams

dose_review = ct.groupby("patient_id").filter(lambda g: g['dose_msv'].sum() >= 100)  # TODO (README 2.3 step 2): every CT of each patient whose doses add up to 100 mSv or more, with filter
print("rows:", len(dose_review))
print("patients:", dose_review["patient_id"].nunique())
display(dose_review)

row_mask = ct[ct["dose_msv"] >= 100]  # TODO (README 2.3 step 3): the row mask that tests each scan instead
print("row mask rows:", len(row_mask))

fast_review = ct[ct.groupby("patient_id")["dose_msv"].transform("sum") >= 100]  # TODO (README 2.3 step 4): the same rows from a transform mask
print("same rows as filter:", fast_review.equals(dose_review))
display(fast_review)
# TODO (README 2.3 step 5): save exam_id, patient_id, and dose_msv to CT_DOSE_PATH
dose_review[["exam_id", "patient_id", "dose_msv"]].to_csv(CT_DOSE_PATH, index=False)

dose_review.head()

### 2.4 Pull each site's three slowest STAT reports

> **Checkpoint: `output/slowest_stat_reports.csv`**
> 24 rows under the header `exam_id,site,modality,report_hr`.

In [ ]:
slowest = stat.groupby("site").apply(lambda g: g.nlargest(3, "report_hr"), include_groups = False)  # TODO (README 2.4 step 1): each site's three slowest STAT reports, with apply
print("rows:", len(slowest))
print("index levels:", slowest.index.nlevels)

# TODO (README 2.4 step 2): move the index levels back into columns and keep four columns
slowest = slowest.reset_index()
slowest = slowest[["exam_id", "site", "modality", "report_hr"]]
display(slowest)

# TODO (README 2.4 step 3): save slowest to SLOWEST_PATH
slowest.to_csv(SLOWEST_PATH, index=False)
slowest

## Task 3: Pivot the report times

### 3.1 Build the pivot table and check it

> **Checkpoint: `output/median_report_pivot.csv`**
> Eight rows, `Downtown` to `Valley`, under the header `site,CT,MRI,Ultrasound,X-ray`; Valley's `MRI` cell stays empty.

In [ ]:
report_pivot = pd.pivot_table(exams, values = "report_hr", index = "site", columns = "modality", aggfunc = "median")  # TODO (README 3.1 step 1): pivot table of median report_hr, sites as rows, modalities as columns
groupby_twin = exams.groupby(["site", "modality"])["report_hr"].median().unstack()  # TODO (README 3.1 step 2): the same medians from a two-key groupby, unstacked
print("pivot matches its groupby twin:", report_pivot.equals(groupby_twin))
print("empty cells:", report_pivot.isna().sum().sum())

# TODO (README 3.1 step 3): save report_pivot to PIVOT_PATH, keeping its site index
report_pivot.to_csv(PIVOT_PATH, index=True)

report_pivot

### 3.2 Count the exams in each pair

This step saves nothing.

In [ ]:
exam_counts = pd.crosstab(exams["site"], exams["modality"])  # TODO (README 3.2): crosstab of exams by site and modality
exam_counts

## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [15]:
from check_assignment import run_checks

_ = run_checks()

Checks: latest from the course repository (main).
[PASS]  2/2  missing by modality: columns
[PASS]  2/2  missing by modality: one row per modality
[PASS]  4/4  missing by modality: shares
[PASS]  4/4  pairplot: saved chart
[PASS]  2/2  site summary: columns
[PASS]  2/2  site summary: one row per site
[PASS]  4/4  site summary: exam_count values
[PASS]  4/4  site summary: report_count values
[PASS]  4/4  site summary: patient_count values
[PASS]  4/4  site summary: total_scan_min values
[PASS]  4/4  site summary: mean_scan_min values
[PASS]  2/2  site and modality: columns
[PASS]  2/2  site and modality: one row per pair
[PASS]  4/4  site and modality: exam_count values
[PASS]  4/4  site and modality: median_wait_hr values
[PASS]  4/4  site and modality: median_report_hr values
[PASS]  2/2  delayed STAT exams: columns
[PASS]  4/4  delayed STAT exams: one row per delayed exam
[PASS]  4/4  delayed STAT exams: original exam values
[PASS]  4/4  delayed STAT exams: peer_median_wait_hr values